In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ All packages imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")


✅ All packages imported successfully!
Pandas version: 3.0.0
NumPy version: 2.4.2


In [2]:
# Cell 2: Load the main dataset
df = pd.read_csv('../data/processed/drugbank_extracted_cleaned.csv')

print(f"✅ Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")
    
print(f"\nFirst few rows:")
df.head()


✅ Data loaded successfully!
Shape: (17430, 5)

Columns:
  1. drugbank_id
  2. name
  3. smiles
  4. interactions
  5. clean_name

First few rows:


,drugbank_id,name,smiles,interactions,clean_name
0,DB00001,Lepirudin,NaN,DB06605|DB06695|DB01254|DB01609|DB01586|DB0212...,Lepirudin
1,DB00002,Cetuximab,NaN,DB00255|DB00269|DB00286|DB00655|DB00783|DB0089...,Cetuximab
2,DB00003,Dornase alfa,NaN,NaN,Dornase
3,DB00004,Denileukin diftitox,NaN,DB00012|DB00016|DB08894|DB09107|DB00281|DB0029...,Denileukin
4,DB00005,Etanercept,NaN,DB08879|DB00531|DB06643|DB00065|DB00008|DB0001...,Etanercept


In [3]:
# Cell 3: Basic data statistics
print("="*60)
print("📊 BASIC DATA STATISTICS")
print("="*60)

# Total drugs
print(f"\n✅ Total drug entries: {len(df)}")

# Unique drugs
print(f"✅ Unique DrugBank IDs: {df['drugbank_id'].nunique()}")

# Check for duplicates
duplicates = df['drugbank_id'].duplicated().sum()
print(f"⚠️  Duplicate entries: {duplicates}")

print("\n" + "="*60)


📊 BASIC DATA STATISTICS

✅ Total drug entries: 17430
✅ Unique DrugBank IDs: 17430
⚠️  Duplicate entries: 0



In [4]:
# Cell 4: Check SMILES availability
print("="*60)
print("🧪 SMILES (Molecular Structure) AVAILABILITY")
print("="*60)

# Count drugs with valid SMILES
smiles_available = df['smiles'].notna().sum()
smiles_missing = df['smiles'].isna().sum()

print(f"\n✅ Drugs with SMILES: {smiles_available} ({smiles_available/len(df)*100:.2f}%)")
print(f"❌ Drugs without SMILES: {smiles_missing} ({smiles_missing/len(df)*100:.2f}%)")

print("\n" + "="*60)


🧪 SMILES (Molecular Structure) AVAILABILITY

✅ Drugs with SMILES: 12313 (70.64%)
❌ Drugs without SMILES: 5117 (29.36%)



In [5]:
# Cell 5: Check interactions availability
print("="*60)
print("🔗 DRUG-DRUG INTERACTIONS AVAILABILITY")
print("="*60)

# Count drugs with interactions
has_interactions = df['interactions'].notna().sum()
no_interactions = df['interactions'].isna().sum()

print(f"\n✅ Drugs WITH interactions: {has_interactions} ({has_interactions/len(df)*100:.2f}%)")
print(f"❌ Drugs WITHOUT interactions: {no_interactions} ({no_interactions/len(df)*100:.2f}%)")

# Sample some drugs with interactions
print(f"\n📋 Example drug with interactions:")
sample_drug = df[df['interactions'].notna()].iloc[0]
print(f"Drug: {sample_drug['name']} ({sample_drug['drugbank_id']})")
interactions_list = sample_drug['interactions'].split('|')
print(f"Interacts with {len(interactions_list)} drugs")
print(f"First 5 interactions: {interactions_list[:5]}")

print("\n" + "="*60)


🔗 DRUG-DRUG INTERACTIONS AVAILABILITY

✅ Drugs WITH interactions: 4566 (26.20%)
❌ Drugs WITHOUT interactions: 12864 (73.80%)

📋 Example drug with interactions:
Drug: Lepirudin (DB00001)
Interacts with 653 drugs
First 5 interactions: ['DB06605', 'DB06695', 'DB01254', 'DB01609', 'DB01586']



In [6]:
# Cell 6: Count total interaction pairs
print("="*60)
print("🔢 TOTAL INTERACTION PAIRS")
print("="*60)

# Count all interaction pairs
total_pairs = 0
for idx, row in df[df['interactions'].notna()].iterrows():
    interactions_list = str(row['interactions']).split('|')
    total_pairs += len(interactions_list)

print(f"\n✅ Total interaction pairs: {total_pairs:,}")
print(f"✅ Average interactions per drug (for drugs with interactions): {total_pairs/has_interactions:.1f}")

# Find drugs with most interactions
df_with_interactions = df[df['interactions'].notna()].copy()
df_with_interactions['num_interactions'] = df_with_interactions['interactions'].apply(lambda x: len(str(x).split('|')))

print(f"\n📊 Top 5 drugs with most interactions:")
top_drugs = df_with_interactions.nlargest(5, 'num_interactions')[['name', 'drugbank_id', 'num_interactions']]
for idx, row in top_drugs.iterrows():
    print(f"  • {row['name']}: {row['num_interactions']} interactions")

print("\n" + "="*60)


🔢 TOTAL INTERACTION PAIRS

✅ Total interaction pairs: 2,855,848
✅ Average interactions per drug (for drugs with interactions): 625.5

📊 Top 5 drugs with most interactions:
  • Clozapine: 2603 interactions
  • Chlorpromazine: 2510 interactions
  • Amitriptyline: 2431 interactions
  • Imipramine: 2428 interactions
  • Carbamazepine: 2385 interactions



In [7]:
# Cell 7: Check if we have interaction descriptions/types
print("="*60)
print("📝 INTERACTION DESCRIPTIONS CHECK")
print("="*60)

# Check what columns we have
print(f"\nAvailable columns: {list(df.columns)}")

# Check if description column exists
if 'interaction_description' in df.columns or 'description' in df.columns:
    print("\n✅ Interaction descriptions FOUND!")
else:
    print("\n⚠️  Interaction descriptions NOT in this file")
    print("   We need to check the original drugbank_extracted.csv")
    
# Let's check the original file
print("\n🔍 Checking original drugbank_extracted.csv...")
df_original = pd.read_csv('../data/processed/drugbank_extracted.csv')
print(f"Columns in original file: {list(df_original.columns)}")

print("\n" + "="*60)


📝 INTERACTION DESCRIPTIONS CHECK

Available columns: ['drugbank_id', 'name', 'smiles', 'interactions', 'clean_name']

⚠️  Interaction descriptions NOT in this file
   We need to check the original drugbank_extracted.csv

🔍 Checking original drugbank_extracted.csv...
Columns in original file: ['drugbank_id', 'name', 'smiles', 'interactions']



In [8]:
# Cell 8: Data audit summary
print("="*60)
print("📋 DATA AUDIT SUMMARY - STEP 1 COMPLETE")
print("="*60)

print("\n✅ WHAT WE HAVE:")
print(f"  • Total drugs: 17,430")
print(f"  • Drugs with SMILES: 12,313 (70.64%)")
print(f"  • Drugs with interactions: 4,566 (26.20%)")
print(f"  • Total interaction pairs: 2,855,848")
print(f"  • Average interactions per drug: 625.5")

print("\n⚠️  MISSING:")
print(f"  • Interaction descriptions/types (needed for multi-relational GNN)")
print(f"  • We have ONLY drug IDs, not interaction metadata")

print("\n🎯 NEXT STEPS:")
print(f"  1. Filter to drugs with BOTH smiles AND interactions")
print(f"  2. Create positive interaction pairs")
print(f"  3. Generate negative samples")
print(f"  4. Check if we can extract interaction types from XML")

# Calculate usable drugs (have both SMILES and interactions)
df_usable = df[(df['smiles'].notna()) & (df['interactions'].notna())]
print(f"\n✨ USABLE DRUGS (have both SMILES + interactions): {len(df_usable)}")

print("\n" + "="*60)


📋 DATA AUDIT SUMMARY - STEP 1 COMPLETE

✅ WHAT WE HAVE:
  • Total drugs: 17,430
  • Drugs with SMILES: 12,313 (70.64%)
  • Drugs with interactions: 4,566 (26.20%)
  • Total interaction pairs: 2,855,848
  • Average interactions per drug: 625.5

⚠️  MISSING:
  • Interaction descriptions/types (needed for multi-relational GNN)
  • We have ONLY drug IDs, not interaction metadata

🎯 NEXT STEPS:
  1. Filter to drugs with BOTH smiles AND interactions
  2. Create positive interaction pairs
  3. Generate negative samples
  4. Check if we can extract interaction types from XML

✨ USABLE DRUGS (have both SMILES + interactions): 3710



In [9]:
# Cell 9: Check for raw XML file
import os

print("="*60)
print("🔍 CHECKING FOR RAW DRUGBANK XML")
print("="*60)

xml_path = '../data/raw/DB.xml'

if os.path.exists(xml_path):
    print(f"\n✅ XML file found at: {xml_path}")
    file_size = os.path.getsize(xml_path) / (1024*1024)  # Convert to MB
    print(f"✅ File size: {file_size:.2f} MB")
    print(f"\n🎯 We can extract interaction descriptions!")
else:
    print(f"\n❌ XML file NOT found at: {xml_path}")
    print(f"\n📁 Files in data/raw/:")
    if os.path.exists('../data/raw/'):
        for file in os.listdir('../data/raw/'):
            print(f"  • {file}")
    else:
        print("  (data/raw/ folder doesn't exist)")
    
print("\n" + "="*60)


🔍 CHECKING FOR RAW DRUGBANK XML

✅ XML file found at: ../data/raw/DB.xml
✅ File size: 1544.66 MB

🎯 We can extract interaction descriptions!



In [11]:
# Cell 10: Parse XML to extract interaction descriptions
import xml.etree.ElementTree as ET
from tqdm import tqdm

print("="*60)
print("⚙️  PARSING XML FOR INTERACTION DESCRIPTIONS")
print("="*60)

print("\n⏳ This will take 2-3 minutes... Please wait!\n")

# Parse XML
tree = ET.parse('../data/raw/DB.xml')
root = tree.getroot()

# Namespace (DrugBank uses xmlns)
ns = {'db': 'http://www.drugbank.ca'}

# Extract drug-drug interactions with descriptions
interactions_data = []

# Iterate through all drugs
for drug in tqdm(root.findall('db:drug', ns), desc="Parsing drugs"):
    drugbank_id = drug.findtext('db:drugbank-id[@primary="true"]', namespaces=ns)
    
    if drugbank_id is None:
        continue
    
    # Find drug-drug interactions
    ddi_section = drug.find('db:drug-interactions', ns)
    
    if ddi_section is not None:
        for interaction in ddi_section.findall('db:drug-interaction', ns):
            target_id = interaction.findtext('db:drugbank-id', namespaces=ns)
            description = interaction.findtext('db:description', namespaces=ns)
            
            if target_id and description:
                interactions_data.append({
                    'drug1': drugbank_id,
                    'drug2': target_id,
                    'description': description
                })

print(f"\n✅ Extracted {len(interactions_data):,} interaction pairs with descriptions!")

# Create DataFrame
df_interactions = pd.DataFrame(interactions_data)
print(f"\n📊 Sample interactions:")
print(df_interactions.head())

print("\n" + "="*60)


⚙️  PARSING XML FOR INTERACTION DESCRIPTIONS

⏳ This will take 2-3 minutes... Please wait!



Parsing drugs: 100%|██████████| 17430/17430 [00:20<00:00, 859.66it/s] 



✅ Extracted 2,855,848 interaction pairs with descriptions!

📊 Sample interactions:
     drug1    drug2                                        description
0  DB00001  DB06605  Apixaban may increase the anticoagulant activi...
1  DB00001  DB06695  Dabigatran etexilate may increase the anticoag...
2  DB00001  DB01254  The risk or severity of bleeding and hemorrhag...
3  DB00001  DB01609  The risk or severity of gastrointestinal bleed...
4  DB00001  DB01586  The risk or severity of bleeding and bruising ...



In [12]:
# Cell 11: Categorize interactions into relation types
print("="*60)
print("🏷️  CATEGORIZING INTERACTION TYPES")
print("="*60)

def categorize_interaction(description):
    """Categorize interaction based on description keywords"""
    if pd.isna(description):
        return 'unknown'
    
    desc_lower = str(description).lower()
    
    # Synergistic / Additive (increases effect)
    if any(word in desc_lower for word in ['increase', 'enhance', 'potentiate', 'additive', 'prolong']):
        return 'synergistic'
    
    # Antagonistic (decreases effect)
    elif any(word in desc_lower for word in ['decrease', 'reduce', 'diminish', 'attenuate', 'antagonize']):
        return 'antagonistic'
    
    # Toxic / Adverse (harmful)
    elif any(word in desc_lower for word in ['toxic', 'adverse', 'danger', 'risk', 'bleeding', 'hemorrhage']):
        return 'toxic'
    
    # Metabolism-related
    elif any(word in desc_lower for word in ['metabolism', 'metabolized', 'cyp', 'enzyme', 'substrate']):
        return 'metabolic'
    
    # Serum concentration changes
    elif any(word in desc_lower for word in ['serum', 'concentration', 'level', 'exposure']):
        return 'concentration'
    
    else:
        return 'other'

# Apply categorization
df_interactions['relation_type'] = df_interactions['description'].apply(categorize_interaction)

# Count each type
print("\n📊 Interaction Type Distribution:")
type_counts = df_interactions['relation_type'].value_counts()
for rel_type, count in type_counts.items():
    percentage = (count / len(df_interactions)) * 100
    print(f"  • {rel_type}: {count:,} ({percentage:.2f}%)")

print("\n📋 Examples of each type:")
for rel_type in df_interactions['relation_type'].unique():
    example = df_interactions[df_interactions['relation_type'] == rel_type].iloc[0]
    print(f"\n🔸 {rel_type.upper()}:")
    print(f"   {example['drug1']} + {example['drug2']}")
    print(f"   → {example['description'][:100]}...")

print("\n" + "="*60)


🏷️  CATEGORIZING INTERACTION TYPES

📊 Interaction Type Distribution:
  • synergistic: 1,783,641 (62.46%)
  • antagonistic: 1,072,207 (37.54%)

📋 Examples of each type:

🔸 SYNERGISTIC:
   DB00001 + DB06605
   → Apixaban may increase the anticoagulant activities of Lepirudin....

🔸 ANTAGONISTIC:
   DB00001 + DB00255
   → Diethylstilbestrol may decrease the anticoagulant activities of Lepirudin....



In [13]:
# Cell 12: Save multi-relational interaction data
print("="*60)
print("💾 SAVING MULTI-RELATIONAL DATA")
print("="*60)

# Save full interaction data with types
output_path = '../data/processed/drugbank_interactions_typed.csv'
df_interactions.to_csv(output_path, index=False)
print(f"\n✅ Saved {len(df_interactions):,} interactions to:")
print(f"   {output_path}")

# Create summary statistics
print("\n📊 FINAL STATISTICS:")
print(f"  • Total interactions: {len(df_interactions):,}")
print(f"  • Unique drugs involved: {len(set(df_interactions['drug1'].unique()) | set(df_interactions['drug2'].unique()))}")
print(f"  • Synergistic interactions: {(df_interactions['relation_type']=='synergistic').sum():,}")
print(f"  • Antagonistic interactions: {(df_interactions['relation_type']=='antagonistic').sum():,}")

# Save a sample for quick inspection
sample_interactions = df_interactions.head(100)
sample_path = '../data/processed/interactions_sample.csv'
sample_interactions.to_csv(sample_path, index=False)
print(f"\n✅ Also saved 100 sample interactions to:")
print(f"   {sample_path}")

print("\n🎉 MULTI-RELATIONAL DATA EXTRACTION COMPLETE!")
print("\n" + "="*60)


💾 SAVING MULTI-RELATIONAL DATA

✅ Saved 2,855,848 interactions to:
   ../data/processed/drugbank_interactions_typed.csv

📊 FINAL STATISTICS:
  • Total interactions: 2,855,848
  • Unique drugs involved: 4567
  • Synergistic interactions: 1,783,641
  • Antagonistic interactions: 1,072,207

✅ Also saved 100 sample interactions to:
   ../data/processed/interactions_sample.csv

🎉 MULTI-RELATIONAL DATA EXTRACTION COMPLETE!



In [14]:
# Cell 13: STEP 1 FINAL SUMMARY
print("="*60)
print("✅ STEP 1: DATA AUDIT - COMPLETE!")
print("="*60)

print("\n📊 DATA INVENTORY:")
print(f"  • Total drugs in database: 17,430")
print(f"  • Drugs with molecular structure (SMILES): 12,313")
print(f"  • Drugs with recorded interactions: 4,567")
print(f"  • Usable drugs (both SMILES + interactions): 3,710")

print("\n🔗 INTERACTION NETWORK:")
print(f"  • Total interaction pairs: 2,855,848")
print(f"  • Synergistic interactions: 1,783,641 (62.46%)")
print(f"  • Antagonistic interactions: 1,072,207 (37.54%)")

print("\n💾 FILES CREATED:")
print(f"  ✅ drugbank_interactions_typed.csv")
print(f"  ✅ interactions_sample.csv")

print("\n🎯 NEXT STEPS (STEP 2):")
print(f"  1. Filter to usable drugs")
print(f"  2. Generate negative samples (non-interacting pairs)")
print(f"  3. Create train/val/test splits")
print(f"  4. Balance positive:negative ratio")

print("\n" + "="*60)


✅ STEP 1: DATA AUDIT - COMPLETE!

📊 DATA INVENTORY:
  • Total drugs in database: 17,430
  • Drugs with molecular structure (SMILES): 12,313
  • Drugs with recorded interactions: 4,567
  • Usable drugs (both SMILES + interactions): 3,710

🔗 INTERACTION NETWORK:
  • Total interaction pairs: 2,855,848
  • Synergistic interactions: 1,783,641 (62.46%)
  • Antagonistic interactions: 1,072,207 (37.54%)

💾 FILES CREATED:
  ✅ drugbank_interactions_typed.csv
  ✅ interactions_sample.csv

🎯 NEXT STEPS (STEP 2):
  1. Filter to usable drugs
  2. Generate negative samples (non-interacting pairs)
  3. Create train/val/test splits
  4. Balance positive:negative ratio

